# Data preparation: Household Pulse Survey, weeks 22–34 (2021)

Replication of Shafer et al. (2022), *Association of the Implementation of Child Tax Credit Advance Payments With Food Insufficiency in US Households*.

This notebook builds the analysis files from the raw Census downloads using the functions in `modules/`:

1. Download each week's CSV zip (optional; or place the zips in `data/raw/` yourself)
2. Unzip them
3. Check which variables each week contains
4. Stack the weeks and build derived variables
5. Save the processed files and run the first replication checks

**Raw data:** [Census 2021 Household Pulse PUF page](https://www.census.gov/programs-surveys/household-pulse-survey/data/datasets.2021.html), weeks 22–34, CSV format. Leave each zip named as Census names it (e.g. `HPS_Week22_PUF_CSV.zip`).

## Setup

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable whether the notebook runs from the repo root or notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from modules import get_data, prepare_pulse

print("Repo root:", ROOT)
print("Raw data folder:", get_data.RAW_DIR)

## 1. Download the raw zips (optional)

Set `DOWNLOAD = True` to fetch the 13 weekly zips from Census (about 1 GB in total). Files already in `data/raw/` are skipped. If you downloaded the zips manually, leave this `False`.

In [ ]:
DOWNLOAD = False

if DOWNLOAD:
    failed = get_data.download_pulse()

## 2. Unzip each week into `data/raw/HPS_Week<NN>_PUF_CSV/`

In [ ]:
status = get_data.unzip_pulse()
missing_zips = [w for w, s in status.items() if s == "MISSING ZIP"]
assert not missing_zips, f"Missing zips for weeks {missing_zips}: download them before continuing."

## 3. Variable check across weeks

Which source column each harmonized variable comes from in each week. Names change between phases (e.g. `EGENDER` → `EGENID_BIRTH` and `SPNDSRC5` → `SPND_SRC5` in week 34). Any `— missing —` cell needs a decision before modeling.

In [ ]:
report = prepare_pulse.variable_report()
missing = report.columns[(report == "— missing —").any()]
print("Variables missing in at least one week:", list(missing) or "none")
report[missing] if len(missing) else report

## 4. Stack weeks and build derived variables

In [ ]:
data, missing_by_week = prepare_pulse.build_analysis_file()
print(data.shape)
data[["WEEK", "food_insuf", "kids", "post", "kids_post", "age", "female", "w"]].head()

### Missingness by week (key variables)

Share of rows missing each variable in each week. Large jumps at week 34 would point to the Phase 3.2 questionnaire change.

In [ ]:
key = ["food_insuf", "INCOME", "female", "EEDUC", "ANYWORK", "SNAP_YN", "UI_SPND", "STIM_SPND"]
key = [c for c in key if c in data]
(data.groupby("WEEK")[key].apply(lambda g: g.isna().mean()) * 100).round(1)

## 5. Replicate weights (for Extension b)

Stacked separately as 32-bit floats; each week is checked for a 1:1 match on `SCRAM`.

In [ ]:
repwgt = prepare_pulse.build_repwgt_file()
repwgt.shape

## 6. Save processed files

In [ ]:
prepare_pulse.save(data, "pulse_w22_34")
prepare_pulse.save(repwgt, "repwgt_w22_34")

## 7. First replication checks

Targets from Shafer et al. (2022): 667,836 respondents under 65 before exclusions; weighted food insufficiency among households with children of 14.3% in week 33 and 9.9% in week 34.

In [ ]:
prepare_pulse.replication_checks(data)